In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
import string
import os

In [3]:
# Load data
property_df = pd.read_csv('QGIS_df_w_CMCI.csv')  # property DataFrame
print(property_df.columns)

Index(['Unnamed: 0', 'Location', 'URL', 'SKU', 'Title', 'Price', 'Category',
       'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area',
       'Description', 'Agent_Name', 'Agent_Link', 'Agent_Verification', 'File',
       'Latitude', 'Longitude', 'LRTHubName', 'LRTHubDist', 'BusHubName',
       'BusHubDist', 'HospitalHubName', 'HospitalHubDist', 'MallHubName',
       'MallHubDist', 'SchoolHubName', 'SchoolHubDist', 'BusCount',
       'HospitalCount', 'MallCount', 'SchoolCount', 'Concat', 'Property Type',
       'Matched_City', 'PROVINCE / LGU', 'employment_generation',
       'financial_deepening', 'local_economy_growth', 'local_economy_size',
       'presence_of_business_and_professional_organizations',
       'safety_compliant_business'],
      dtype='object')


In [4]:
atm_df = pd.read_csv("20250305 Updated Properties with ATM.csv")
atm_df.head()

,fid,Location,URL,SKU,Title,Price,Category,Num_Bedroo,Num_Bathro,Floor_Area,...,MallHubDist,SchoolHubName,SchoolHubDist,BusCount,HospitalCount,MallCount,SchoolCount,ATMHubName,ATMHubDist,ATMCount
0,1,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo-resi...,HO5CEE34F0060ECPH,7.2M single attached house and lot for sale at...,7200000.0,house,3.0,2.0,147.0,...,2.509902,w873447606,0.342505,0,0,0,5,948,1.660980,0
1,2,Caloocan,https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66D1B50AECE7EPH,Caloocan City RFO Condo beside Lrt Monumento,3100000.0,condo,1.0,1.0,21.7,...,0.708891,w40941344,0.295216,14,7,3,15,26,0.008137,8
2,3,"Amparo, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,LA66C596CFA8463PH,"850sqm Vacant Lot in Makabud Street, Amparo, N...",17000000.0,land,0.0,0.0,0.0,...,2.509902,w873447606,0.342505,0,0,0,5,948,1.660980,0
3,4,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C1D00AA6DD1PH,Pre Selling 2BR 97sqm in Caloocan City near Ar...,11900000.0,condo,3.0,2.0,97.0,...,0.698302,w40941344,0.414120,15,10,2,15,625,0.665267,12
4,5,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C6B155BF32BPH,3BR 97sqm Pre Selling in Caloocan City near Sq...,11900000.0,condo,3.0,2.0,97.0,...,0.698302,w40941344,0.414120,15,10,2,15,625,0.665267,12


In [16]:
property_sku_list = list(property_df["SKU"])
atm_sku_list = list(atm_df["SKU"])

not_in_qgis = list(set(atm_sku_list) - set(property_sku_list))
atm_df[atm_df["SKU"].isin(not_in_qgis)]

,fid,Location,URL,SKU,Title,Price,Category,Num_Bedroo,Num_Bathro,Floor_Area,...,MallHubDist,SchoolHubName,SchoolHubDist,BusCount,HospitalCount,MallCount,SchoolCount,ATMHubName,ATMHubDist,ATMCount
127,128,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo/read...,HO5F13C22D8F8A2PH,Ready to move in house FOR SALE in Amparo Calo...,6500000.0,house,3.0,2.0,120.89,...,2.509902,w873447606,0.342505,0,0,0,5,948,1.660980,0
148,149,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/victory-hei...,HO66B1E25A5D272PH,Victory Heights North Caloocan Preselling Town...,3990000.0,house,3.0,3.0,92.00,...,2.509902,w873447606,0.342505,0,0,0,5,948,1.660980,0
183,184,"Deparo, Caloocan",https://www.lamudi.com.ph/projects/tres-marias...,HO66B5C6F3A44A9PH,Deparo North Caloocan Brandnew House and Lot 1...,11250000.0,house,4.0,3.0,165.00,...,3.145887,n10889215132,0.223600,0,0,0,12,403,1.091750,0
194,195,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo-subd...,NO538HO48POTINTRESPH,Ready to move in house FOR SALE in Amparo Calo...,7000000.0,house,3.0,2.0,98.12,...,2.509902,w873447606,0.342505,0,0,0,5,948,1.660980,0
208,209,"Amparo, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,HO6062D7B0ADF84PH,"Unit 2, 3 Bedroom House and Lot for sale in Ma...",4080000.0,house,3.0,2.0,83.70,...,2.509902,w873447606,0.342505,0,0,0,5,948,1.660980,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80105,80106,"Karuhatan, Valenzuela",https://www.lamudi.com.ph/buy/metro-manila/val...,LA664F00C091172PH,"Industrial Properties FOR SALE in, Brgy. Karuh...",20000000.0,land,0.0,0.0,0.00,...,0.461623,w171611933,0.115401,12,3,2,11,617,0.127978,5
80107,80108,"Ugong, Valenzuela",https://www.lamudi.com.ph/buy/metro-manila/val...,AP6041AA6F9F12DPH,Apartment in Valenzuela City,20900000.0,apartment,4.0,3.0,0.00,...,1.397397,w569044803,0.037824,0,0,0,7,687,1.750623,0
80137,80138,"Canumay, Valenzuela",https://www.lamudi.com.ph/buy/metro-manila/val...,HO667519B114C34PH,Preselling 2 Storey 3-4 BR House and Lot in Ca...,8275000.0,house,3.0,2.0,84.00,...,2.546867,w553033732,0.570302,0,0,0,6,309,0.428383,3
80152,80153,"Malanday, Valenzuela",https://www.lamudi.com.ph/buy/metro-manila/val...,HO66751D8F03B54PH,Preselling 3 Storey 4 Bedroom Townhouse in Mal...,6650000.0,house,4.0,3.0,106.00,...,2.582603,w654688521,0.661036,6,2,0,9,417,0.641290,3


In [6]:
len(atm_df) - len(property_df)

8204

In [ ]:
merged_df = property_df.merge(atm_df[["SKU", "ATMHubName", "ATMHubDist", "ATMCount"]], on="SKU", how="left")
merged_df = merged_df.iloc[:,1:]
merged_df.head()
# merged_df.to_csv("merged_df.csv", encoding="utf-8-sig")